# Batch Photometry Demo

This notebook demonstrates the `spxquery.batch` module for extracting multi-source aperture photometry from SPHEREx full-frame images.

**Workflow:**
1. Prepare a source catalog (CSV with `targetid`, `ra`, `dec`)
2. Query IRSA TAP for full-frame images covering a sky region
3. Download the images
4. Extract photometry for all sources in each image's FOV
5. Aggregate into per-source light curves
6. Inspect results

In [ ]:
import sys
from pathlib import Path

# Ensure spxquery from this repo is importable
sys.path.insert(0, str(Path("../../src").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from spxquery.batch import BatchConfig, BatchPipeline
from spxquery.batch.config import load_catalog
from spxquery.core.config import PhotometryConfig, Source

## 1. Prepare a Source Catalog

We create a small catalog of known sources around the NEP (North Ecliptic Pole) region. In practice, you would load an existing catalog from DESI, SDSS, or your own observations.

In [ ]:
# Demo directory and output
DEMO_DIR = Path(".").resolve()
OUTPUT_DIR = DEMO_DIR / "batch_output"

# Create a small catalog of sources near the NEP
# These are real coordinates from the DESI NEP field
catalog_data = {
    "targetid": [
        39633451355209872,
        39633446326241333,
        39633456308686512,
        39633453871796114,
        39633461169884285,
        39633465875891054,
        39633443780300215,
        39633470535763603,
    ],
    "ra": [266.10, 268.40, 269.90, 270.50, 271.20, 271.80, 272.50, 273.30],
    "dec": [66.20, 66.50, 66.80, 66.30, 67.00, 66.60, 66.90, 67.30],
}

catalog_df = pd.DataFrame(catalog_data)
catalog_path = DEMO_DIR / "demo_catalog.csv"
catalog_df.to_csv(catalog_path, index=False)

print(f"Catalog: {len(catalog_df)} sources")
print(f"  RA range:  {catalog_df['ra'].min():.2f} - {catalog_df['ra'].max():.2f}")
print(f"  Dec range: {catalog_df['dec'].min():.2f} - {catalog_df['dec'].max():.2f}")
catalog_df

## 2. Configure and Run the Pipeline

`BatchConfig` defines the sky region, catalog path, and processing parameters. The pipeline has four stages:

| Stage | Function | Description |
|-------|----------|-------------|
| 1 | `run_query()` | Query IRSA TAP for full-frame images |
| 2 | `run_download()` | Download FITS files (no cutouts) |
| 3 | `run_extract()` | Multi-source per-image photometry |
| 4 | `run_aggregate()` | Per-source light curve CSVs |

**Safety gate:** If the query returns more than `max_images` (default 500), it raises an error to prevent accidental large downloads.

In [ ]:
# Use a small radius to keep this demo fast (~3-10 images)
config = BatchConfig(
    center_ra=270.0,
    center_dec=66.6,
    radius=0.5,                       # Small region for demo
    catalog_path=catalog_path,
    output_dir=OUTPUT_DIR,
    coverage_mode="any",              # INTERSECTS — images touching the circle
    bands=None,                       # All bands
    max_images=20,                    # Safety cap for demo
    max_download_workers=4,
    max_extract_workers=4,            # Fewer workers for demo
    photometry=PhotometryConfig(
        aperture_method="fwhm",
        fwhm_multiplier=2.5,
        background_method="window",
        window_size=30,
        subtract_zodi=True,
    ),
)

print(f"Region: RA={config.center_ra}, Dec={config.center_dec}, radius={config.radius} deg")
print(f"Output: {config.output_dir}")

In [ ]:
pipeline = BatchPipeline(config)

### Stage 1: Query

In [ ]:
query_results = pipeline.run_query()

print(f"\nFound {len(query_results)} observations")
print("Bands:")
for band, count in sorted(query_results.band_counts.items()):
    print(f"  {band}: {count} images")

### Stage 2: Download

Full-frame images (~71.6 MB each). This may take a few minutes depending on your connection.

In [ ]:
download_results = pipeline.run_download(skip_existing=True)

n_success = sum(1 for r in download_results if r.success)
print(f"\nDownloaded {n_success}/{len(download_results)} images")

### Stage 3: Extract Photometry

For each image, the pipeline:
1. Reads the MEF file once
2. Projects all catalog sources to pixel coordinates via WCS
3. Filters to sources within the field of view
4. Extracts aperture photometry for each in-FOV source
5. Saves results as a per-image CSV (incremental / resumable)

In [ ]:
n_new = pipeline.run_extract(skip_existing=True)
print(f"\nExtracted {n_new} per-image CSVs")

### Stage 4: Aggregate Light Curves

Combines per-image CSVs into per-source light curves using a memory-efficient bucket-based approach.

In [ ]:
n_sources = pipeline.run_aggregate(clean=True)
print(f"\nCreated {n_sources} light curve files")

## 3. Inspect Results

In [ ]:
# List all light curve files
lc_files = sorted(config.lightcurve_dir.glob("*.csv"))
print(f"Light curves: {len(lc_files)} files")

# Show directory structure
print(f"\nOutput structure:")
print(f"  {config.output_dir}/")
print(f"    images/          — {len(list(config.image_dir.rglob('*.fits')))} FITS files")
print(f"    per_image/       — {len(list(config.per_image_dir.glob('*.csv')))} CSV files")
print(f"    lightcurves/     — {len(lc_files)} CSV files")

In [ ]:
# Load and display a sample light curve
if lc_files:
    sample = lc_files[0]
    lc = pd.read_csv(sample)
    print(f"Sample: {sample.name} — {len(lc)} observations")
    display(lc.head(10))

In [ ]:
# Quick spectral view: flux vs wavelength for sources with multiple observations
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
axes = axes.ravel()

plotted = 0
for lc_file in lc_files:
    lc = pd.read_csv(lc_file)
    if len(lc) < 2:
        continue
    if plotted >= 4:
        break

    ax = axes[plotted]
    target_id = lc_file.stem

    for band, grp in lc.groupby("band"):
        ax.errorbar(
            grp["wavelength"],
            grp["flux"],
            yerr=grp["flux_error"],
            fmt="o",
            label=band,
            markersize=4,
            capsize=2,
        )

    ax.set_title(f"{target_id}", fontsize=9)
    ax.set_xlabel(r"$\lambda$ [$\mu$m]")
    if plotted % 2 == 0:
        ax.set_ylabel(r"Flux [$\mu$Jy]")
    ax.legend(fontsize=7)
    ax.set_yscale("log")
    plotted += 1

# Hide unused axes
for i in range(plotted, 4):
    axes[i].set_visible(False)

fig.suptitle("SPHEREx Batch Photometry — Sample Light Curves", fontsize=12)
plt.show()

## 4. Alternative: One-Line Execution

For production use, you can run the entire pipeline with a single function call:

In [ ]:
from spxquery.batch import run_batch

# This runs all four stages: query -> download -> extract -> aggregate
# Uncomment to execute:

# pipeline = run_batch(
#     catalog="demo_catalog.csv",
#     center_ra=270.0,
#     center_dec=66.6,
#     radius=0.5,
#     output_dir="batch_output_v2",
#     max_images=20,
#     max_extract_workers=4,
# )

## Notes

- **Incremental processing:** Stages 3 and 4 are resumable. If interrupted, re-running `run_extract(skip_existing=True)` will skip already-processed images.
- **Size gate:** Set `max_images` to prevent accidental large downloads. Increase it if you need more images.
- **Coverage modes:** `"any"` finds images that partially overlap the region; `"full"` requires the entire image to be inside the circle.
- **Memory:** The aggregation uses a bucket-based approach that never loads the full dataset into memory.